### **Analisando Artigos - Importando pelo findpapers**

In [1]:
# Importando Blibiotecas
import os
import json
import pandas as pd
import requests
import time
import json
from pathlib import Path

In [ ]:
# Definindo a query
query = (
    "([videofluoroscopy] OR [videofluoroscopic swallowing study] OR "
    "[videofluoroscopic swallowing studies] OR [VFSS] OR "
    "[modified barium swallow] OR [MBS] OR [barium swallow study]) "
    "AND "
    "([artificial intelligence] OR [machine learning] OR [deep learning] OR "
    "[neural network] OR [convolutional neural network] OR [CNN] OR "
    "[computer vision] OR [image recognition] OR [video analysis] OR "
    "[object detection] OR [pose estimation] OR [optical flow] OR "
    "[segmentation] OR [tracking] OR [classification] OR [prediction]) "
    "AND "
    "([swallowing] OR [deglutition] OR [dysphagia] OR "
    "[aspiration] OR [penetration] OR [pharyngeal] OR [laryngeal] OR "
    "[hyoid] OR [epiglottis] OR [bolus] OR "
    "[upper esophageal sphincter] OR [UES] OR "
    "[swallowing kinematics] OR [swallowing timing] OR [swallowing events] OR " 
    "[cervical] OR [vertebrae] OR [mandibule] OR [residue] OR "
    "[deglutition disorders] OR [laryngeal closure])"
)

In [3]:
# Fazendo a busca

output_json = r"C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\vfss_ai_papers.json"

ret = os.system(
    f'findpapers search "{output_json}" -q "{query}"'
)

if ret:
    print("ERRO no comando")
else:
    print("Seleção Concluída!")

Seleção Concluída!


In [4]:
# Aumentando as informações do Json encontrado.

def buscar_doi_pubmed(titulo: str) -> tuple[str | None, str | None]:
    """Busca DOI e URL no PubMed pela API Entrez usando o título do artigo."""
    try:
        # 1. Busca o PMID pelo título
        search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
        r = requests.get(search_url, params={
            "db": "pubmed", "term": titulo, "retmode": "json", "retmax": 1
        }, timeout=10)
        ids = r.json().get("esearchresult", {}).get("idlist", [])
        if not ids:
            return None, None

        pmid = ids[0]

        # 2. Busca os detalhes pelo PMID
        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
        r2 = requests.get(fetch_url, params={
            "db": "pubmed", "id": pmid, "retmode": "json"
        }, timeout=10)
        result = r2.json().get("result", {}).get(pmid, {})

        doi = None
        for id_obj in result.get("articleids", []):
            if id_obj.get("idtype") == "doi":
                doi = id_obj.get("value")
                break

        url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else None
        return doi, url

    except Exception as e:
        return None, None

def enriquecer_json(search_json_path: str, delay: float = 0.4):
    """
    Percorre todos os artigos do JSON de busca e preenche doi + urls
    nos que estiverem vazios, consultando o PubMed.
    Salva o JSON enriquecido no mesmo arquivo.
    """
    with open(search_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    atualizados = 0
    sem_resultado = []

    for i, p in enumerate(data["papers"]):
        tem_doi  = bool(p.get("doi"))
        tem_urls = bool(p.get("urls"))

        if tem_doi and tem_urls:
            continue  # já está completo

        titulo = p.get("title", "")
        print(f"[{i+1}/{len(data['papers'])}] Buscando: {titulo[:70]}...")

        doi, url = buscar_doi_pubmed(titulo)

        if doi:
            p["doi"] = doi
            atualizados += 1
            print(f"  ✅ DOI encontrado: {doi}")
        else:
            sem_resultado.append(titulo)
            print(f"  ⚠️  Sem DOI")

        if url:
            urls = list(p.get("urls") or [])
            if url not in urls:
                urls.append(url)
            if doi:
                doi_url = f"http://doi.org/{doi}"
                if doi_url not in urls:
                    urls.append(doi_url)
            p["urls"] = urls

        time.sleep(delay)  # respeita o rate limit da API do PubMed (max 3 req/s)

    with open(search_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"\n✅ Enriquecimento concluído!")
    print(f"   Atualizados : {atualizados}")
    print(f"   Sem resultado: {len(sem_resultado)}")
    if sem_resultado:
        print("\nArtigos sem DOI encontrado:")
        for t in sem_resultado:
            print(f"  - {t}")

# ── Executa ──────────────────────────────────────────────────
enriquecer_json(output_json)

[1/228] Buscando: Expert Consensus Statement on Acoustic Metrics for Swallowing Dysfunct...
  ✅ DOI encontrado: 10.1016/j.jvoice.2026.05.036
[2/228] Buscando: Deep learning for early detection of Zenker's diverticulum based on sw...
  ✅ DOI encontrado: 10.1007/s11548-026-03727-8
[3/228] Buscando: Deep Learning-Based Acoustic Screening for Penetration-Aspiration Even...
  ✅ DOI encontrado: 10.1007/s00455-026-10956-1
[4/228] Buscando: Post-swallowing voice-based aspiration screening in dysphagia using a ...
  ✅ DOI encontrado: 10.1038/s41598-026-53618-w
[5/228] Buscando: SPARNet: A Framework for Airway Invasion Tracking from Fluoroscopic Vi...
  ✅ DOI encontrado: 10.1109/JBHI.2026.3695144
[6/228] Buscando: Swallowing impairment and aspiration risk in clinically stabilized pat...
  ✅ DOI encontrado: 10.3389/fmed.2026.1804250
[7/228] Buscando: YOLO11-based detection of manometry sensors in video-fluoroscopy imagi...
  ✅ DOI encontrado: 10.3389/fradi.2026.1767875
[8/228] Buscando: Objective

In [5]:
# Examinando artigos encontrados

with open(output_json, "r", encoding="utf-8") as f:
    papers = json.load(f)

df = pd.DataFrame(papers["papers"])

print(f"{len(df)} artigos encontrados")
df.head()

228 artigos encontrados


,abstract,authors,categories,citations,comments,databases,doi,keywords,number_of_pages,pages,publication,publication_date,selected,title,urls
0,OBJECTIVE OBJECTIVE Swallowing dysfunction pos...,"[Adrián Castillo-Allendes, Sara W Albert, Jame...",None,None,None,[PubMed],10.1016/j.jvoice.2026.05.036,"[N Dysphagia, N Expert consensus, N Swallowing...",NaN,None,"{'category': 'Journal', 'cite_score': None, 'i...",2026-06-16,None,Expert Consensus Statement on Acoustic Metrics...,"[https://pubmed.ncbi.nlm.nih.gov/42303507/, ht..."
1,PURPOSE OBJECTIVE Patients with Zenker's diver...,"[Daniel Ostler-Mildner, Alissa Jell, Matthias ...",None,None,None,[PubMed],10.1007/s11548-026-03727-8,"[N Dysphagia, N Biomedical acoustics, N Comput...",NaN,None,"{'category': 'Journal', 'cite_score': None, 'i...",2026-06-03,None,Deep learning for early detection of Zenker's ...,"[https://pubmed.ncbi.nlm.nih.gov/42234064/, ht..."
2,To evaluate the feasibility of a smartphone-ba...,"[Yong Jae Na, Jun Hyeok Lee, Eunyoung Choi, Jo...",None,None,None,[PubMed],10.1007/s00455-026-10956-1,"[N Artificial intelligence, N Machine learning...",NaN,None,"{'category': 'Journal', 'cite_score': None, 'i...",2026-06-02,None,Deep Learning-Based Acoustic Screening for Pen...,"[https://pubmed.ncbi.nlm.nih.gov/42228093/, ht..."
3,Dysphagia presents a serious risk of aspiratio...,"[Jung-Min Kim, Min-Seop Kim, Sun-Young Choi, H...",None,None,None,[PubMed],10.1038/s41598-026-53618-w,"[N Voice analysis, N Real-time monitoring, N D...",NaN,None,"{'category': 'Journal', 'cite_score': None, 'i...",2026-05-21,None,Post-swallowing voice-based aspiration screeni...,"[https://pubmed.ncbi.nlm.nih.gov/42168445/, ht..."
4,The videofluoroscopic swallowing study (VFSS) ...,"[Sanjeevi G, Uma Gopalakrishnan, Rahul Krishna...",None,None,None,[PubMed],10.1109/JBHI.2026.3695144,[],NaN,None,"{'category': 'Journal', 'cite_score': None, 'i...",2026-05-19,None,SPARNet: A Framework for Airway Invasion Track...,"[https://pubmed.ncbi.nlm.nih.gov/42154713/, ht..."


In [6]:
# Verificando algumas informações

df.columns.tolist()

cols = [c for c in df.columns if any(
    k in c.lower()
    for k in ["title", "author", "year", "doi", "abstract"]
)]

df[cols].head()

,abstract,authors,doi,title
0,OBJECTIVE OBJECTIVE Swallowing dysfunction pos...,"[Adrián Castillo-Allendes, Sara W Albert, Jame...",10.1016/j.jvoice.2026.05.036,Expert Consensus Statement on Acoustic Metrics...
1,PURPOSE OBJECTIVE Patients with Zenker's diver...,"[Daniel Ostler-Mildner, Alissa Jell, Matthias ...",10.1007/s11548-026-03727-8,Deep learning for early detection of Zenker's ...
2,To evaluate the feasibility of a smartphone-ba...,"[Yong Jae Na, Jun Hyeok Lee, Eunyoung Choi, Jo...",10.1007/s00455-026-10956-1,Deep Learning-Based Acoustic Screening for Pen...
3,Dysphagia presents a serious risk of aspiratio...,"[Jung-Min Kim, Min-Seop Kim, Sun-Young Choi, H...",10.1038/s41598-026-53618-w,Post-swallowing voice-based aspiration screeni...
4,The videofluoroscopic swallowing study (VFSS) ...,"[Sanjeevi G, Uma Gopalakrishnan, Rahul Krishna...",10.1109/JBHI.2026.3695144,SPARNet: A Framework for Airway Invasion Track...


In [7]:
# Salvando informações em um excel

df.to_excel(
    r"..\data\artigos\vfss_ai_papers.xlsx",
    index=False
)

print("Arquivo salvo!")

Arquivo salvo!


### **Analisando Artigos - Importando manualmente pelos sites**

Dados coletados manualmente dos seus respectivos sites, para que o findpapers consiga ler, devemos considerar somente artigos (nada de conferência) e em inglês. Filtros de ano podem ser feitos depois, visto que essa informação é importada.

In [17]:
# Importando Bibliotecas

import glob
import json
import os
from datetime import datetime
 
import pandas as pd
import rispy
from Bio import Medline
from openpyxl import Workbook
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Font
import matplotlib.pyplot as plt
from openpyxl import load_workbook
import textwrap

from itertools import combinations
from rapidfuzz import fuzz
from openpyxl.styles import Font, PatternFill
from openpyxl.formatting.rule import ColorScaleRule
import seaborn as sns

In [18]:
# Constantes para a aplicação

PASTA_ENTRADA = r"C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\query_results"
PASTA_SAIDA = r"C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados"
 
ARQUIVOS_RIS = {
    "Scopus": "ASPEKT_Survey_Scopus_2026_07_11.ris",
    "Web of Science": "ASPEKT_Survey_WebOfScience_2026_07_11.ris",
    "Embase": "ASPEKT_Survey_Embase_2026_07_11.ris",
    "IEEE Xplore": "ASPEKT_Survey_IEEEXplore_2026_07_11.ris",
}
ARQUIVO_PUBMED_NBIB = "ASPEKT_Survey_PubMed_2026_07_11.nbib"

LIMIAR_SIMILARIDADE = 92  # 0-100, quanto maior mais rígido (menos falsos positivos) - artigos duplicados

BASE_PRIMARIA = True  # True = cada artigo conta 1x (na base "vencedora" do dedup para análise exploratoria)
                      # False = artigo conta em TODAS as bases onde foi encontrado

ANO_CORTE = 2015
MANTER_SEM_ANO = True  # True = mantém artigos sem ano identificado (para revisão manual)

In [19]:
# Parseando para o formato do findpapers

# PARSER RIS (Scopus, WoS, Embase, IEEE) -> dict no formato findpapers
def parse_ris_para_findpapers(caminho: str, nome_base: str) -> list[dict]:
    with open(caminho, "r", encoding="utf-8-sig", errors="ignore") as f:
        entradas = rispy.load(f, skip_unknown_tags=True)
 
    papers = []
    for e in entradas:
        autores = e.get("authors") or []
        keywords = e.get("keywords") or []
        doi = (e.get("doi") or "").strip() or None
        titulo = (e.get("title") or e.get("primary_title") or "").strip()
        ano = e.get("year") or e.get("publication_year") or ""
        data_pub = f"{ano}-01-01" if ano else None
 
        urls = []
        if e.get("url"):
            urls.append(e["url"])
        if doi:
            urls.append(f"http://doi.org/{doi}")
 
        papers.append({
            "abstract": (e.get("abstract") or "").strip() or None,
            "authors": autores,
            "categories": None,
            "citations": None,
            "comments": None,
            "databases": [nome_base],
            "doi": doi,
            "keywords": keywords,
            "number_of_pages": None,
            "pages": e.get("start_page") or None,
            "publication": {
                "category": None,
                "cite_score": None,
                "is_potentially_predatory": False,
                "isbn": e.get("isbn") or None,
                "issn": e.get("issn") or None,
                "publisher": e.get("publisher") or None,
                "sjr": None,
                "snip": None,
                "subject_areas": [],
                "title": e.get("journal_name") or e.get("secondary_title") or None,
            },
            "publication_date": data_pub,
            "selected": None,
            "title": titulo,
            "urls": urls,
        })
    return papers
 
# PARSER NBIB/MEDLINE (PubMed) -> dict no formato findpapers
def parse_nbib_para_findpapers(caminho: str) -> list[dict]:
    papers = []
    with open(caminho, encoding="utf-8", errors="ignore") as f:
        registros = Medline.parse(f)
        for r in registros:
            doi = None
            for aid in r.get("AID", []):
                if "[doi]" in aid:
                    doi = aid.replace("[doi]", "").strip()
                    break
 
            pmid = r.get("PMID")
            urls = [f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"] if pmid else []
            if doi:
                urls.append(f"http://doi.org/{doi}")
 
            keywords = list(r.get("OT", [])) + list(r.get("MH", []))
 
            data_pub = None
            dp = r.get("DP")  # ex: "2026 Jun 16" ou "2026"
            if dp:
                ano = dp.split()[0]
                if ano.isdigit():
                    data_pub = f"{ano}-01-01"
 
            papers.append({
                "abstract": r.get("AB") or None,
                "authors": r.get("AU", []),
                "categories": None,
                "citations": None,
                "comments": None,
                "databases": ["PubMed"],
                "doi": doi,
                "keywords": keywords,
                "number_of_pages": None,
                "pages": r.get("PG") or None,
                "publication": {
                    "category": None,
                    "cite_score": None,
                    "is_potentially_predatory": False,
                    "isbn": None,
                    "issn": r.get("IS") or None,
                    "publisher": None,
                    "sjr": None,
                    "snip": None,
                    "subject_areas": [],
                    "title": r.get("JT") or r.get("TA") or None,
                },
                "publication_date": data_pub,
                "selected": None,
                "title": r.get("TI") or "",
                "urls": urls,
            })
    return papers

In [20]:
# Unificando Arquivos
def unificar(pasta_entrada: str) -> dict:
    todos_papers = []
    contagem_por_base = {}
 
    for nome_base, nome_arquivo in ARQUIVOS_RIS.items():
        caminho = os.path.join(pasta_entrada, nome_arquivo)
        if not os.path.exists(caminho):
            print(f"[aviso] não encontrei {caminho}, pulando {nome_base}")
            continue
        papers = parse_ris_para_findpapers(caminho, nome_base)
        print(f"{nome_base}: {len(papers)} artigos lidos de {nome_arquivo}")
        contagem_por_base[nome_base] = len(papers)
        todos_papers.extend(papers)
 
    caminho_pubmed = os.path.join(pasta_entrada, ARQUIVO_PUBMED_NBIB)
    if os.path.exists(caminho_pubmed):
        papers_pubmed = parse_nbib_para_findpapers(caminho_pubmed)
        print(f"PubMed: {len(papers_pubmed)} artigos lidos de {ARQUIVO_PUBMED_NBIB}")
        contagem_por_base["PubMed"] = len(papers_pubmed)
        todos_papers.extend(papers_pubmed)
    else:
        print(f"[aviso] não encontrei {caminho_pubmed}, pulando PubMed")
 
    resultado = {
        "databases": list(contagem_por_base.keys()),
        "limit": None,
        "limit_per_database": None,
        "number_of_papers": len(todos_papers),
        "number_of_papers_by_database": contagem_por_base,
        "papers": todos_papers,
        "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "publication_types": None,
        "query": None,  # cada base teve sua própria query; não cabe um único campo aqui
        "since": None,
        "until": None,
    }
    return resultado

In [21]:
# Gráficos e Exportação
def salvar_json(resultado: dict, caminho_saida: str):
    with open(caminho_saida, "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=2)
    print(f"\nJSON salvo em: {caminho_saida}")
 
 
def salvar_excel_com_grafico(resultado: dict, caminho_saida: str, caminho_png: str):
    linhas = []
    for p in resultado["papers"]:
        linhas.append({
            "titulo": p["title"],
            "abstract": p["abstract"],
            "autores": "; ".join(p["authors"]) if p["authors"] else "",
            "keywords": "; ".join(p["keywords"]) if p["keywords"] else "",
            "doi": p["doi"],
            "revista_evento": p["publication"]["title"],
            "issn": p["publication"]["issn"],
            "data_publicacao": p["publication_date"],
            "base_origem": ", ".join(p["databases"]),
            "url": p["urls"][0] if p["urls"] else "",
        })
    df = pd.DataFrame(linhas)
 
    wb = Workbook()
 
    ws_dados = wb.active
    ws_dados.title = "Artigos"
    ws_dados.append(list(df.columns))
    for cell in ws_dados[1]:
        cell.font = Font(bold=True)
    for row in df.itertuples(index=False):
        ws_dados.append(list(row))
    larguras = [50, 60, 30, 30, 22, 30, 14, 16, 18, 40]
    for i, w in enumerate(larguras, start=1):
        ws_dados.column_dimensions[ws_dados.cell(row=1, column=i).column_letter].width = w
 
    ws_resumo = wb.create_sheet("Resumo")
    ws_resumo.append(["Base", "Nº de artigos"])
    for cell in ws_resumo[1]:
        cell.font = Font(bold=True)
    contagem = resultado["number_of_papers_by_database"]
    for base, qtd in contagem.items():
        ws_resumo.append([base, qtd])
    ws_resumo.append(["TOTAL", resultado["number_of_papers"]])
    ws_resumo["A1"].font = Font(bold=True)
 
    grafico = BarChart()
    grafico.title = "Artigos por base (antes da deduplicação)"
    grafico.y_axis.title = "Nº de artigos"
    grafico.x_axis.title = "Base"
    n = len(contagem)
    dados = Reference(ws_resumo, min_col=2, min_row=1, max_row=1 + n)
    categorias = Reference(ws_resumo, min_col=1, min_row=2, max_row=1 + n)
    grafico.add_data(dados, titles_from_data=True)
    grafico.set_categories(categorias)
    ws_resumo.add_chart(grafico, "D2")
 
    wb.save(caminho_saida)
    print(f"Excel salvo em: {caminho_saida}")
 
    # PNG standalone (para visualização rápida fora do Excel)
    plt.figure(figsize=(7, 4.5))
    plt.bar(list(contagem.keys()), list(contagem.values()), color="#4472C4")
    plt.title("Artigos por base (antes da deduplicação)")
    plt.ylabel("Nº de artigos")
    plt.xticks(rotation=20, ha="right")
    for i, (base, qtd) in enumerate(contagem.items()):
        plt.text(i, qtd, str(qtd), ha="center", va="bottom")
    plt.tight_layout()
    plt.savefig(caminho_png, dpi=150)
    plt.close()
    print(f"Gráfico PNG salvo em: {caminho_png}")

In [22]:
# Salvando Resultados Unificados
resultado = unificar(PASTA_ENTRADA)
salvar_json(resultado, os.path.join(PASTA_SAIDA,"unificado_findpapers.json"))
salvar_excel_com_grafico(resultado, os.path.join(PASTA_SAIDA,"unificado.xlsx"), os.path.join(PASTA_SAIDA,"contagem_por_base.png"))

Scopus: 469 artigos lidos de ASPEKT_Survey_Scopus_2026_07_11.ris
Web of Science: 227 artigos lidos de ASPEKT_Survey_WebOfScience_2026_07_11.ris
Embase: 160 artigos lidos de ASPEKT_Survey_Embase_2026_07_11.ris
IEEE Xplore: 23 artigos lidos de ASPEKT_Survey_IEEEXplore_2026_07_11.ris
PubMed: 30 artigos lidos de ASPEKT_Survey_PubMed_2026_07_11.nbib

JSON salvo em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados\unificado_findpapers.json
Excel salvo em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados\unificado.xlsx
Gráfico PNG salvo em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados\contagem_por_base.png


### **Código de Remoção de Artigos Duplicados**

In [23]:
# Lógica de remoção de artigos duplicados

# Carrega Json com os artigos unificados
def load_json_articles(caminho: str) -> dict:
    with open(caminho, "r", encoding="utf-8") as f:
        return json.load(f)

# Une informações em artigos duplicados
def mesclar_grupo(grupo: list[dict]) -> dict:
    # prioriza o registro com abstract preenchido como "base" da mesclagem
    grupo_ordenado = sorted(grupo, key=lambda p: bool(p.get("abstract")), reverse=True)
    base = dict(grupo_ordenado[0])

    todas_databases = []
    for p in grupo:
        for db in p.get("databases") or []:
            if db not in todas_databases:
                todas_databases.append(db)
    base["databases"] = todas_databases

    todos_autores = []
    for p in grupo:
        for a in p.get("authors") or []:
            if a not in todos_autores:
                todos_autores.append(a)
    if len(todos_autores) >= len(base.get("authors") or []):
        base["authors"] = todos_autores

    todas_keywords = []
    for p in grupo:
        for k in p.get("keywords") or []:
            if k not in todas_keywords:
                todas_keywords.append(k)
    base["keywords"] = todas_keywords

    todas_urls = []
    for p in grupo:
        for u in p.get("urls") or []:
            if u not in todas_urls:
                todas_urls.append(u)
    base["urls"] = todas_urls

    for campo in ("doi", "pages", "number_of_pages", "publication_date"):
        if not base.get(campo):
            for p in grupo:
                if p.get(campo):
                    base[campo] = p[campo]
                    break

    pub_base = base.get("publication") or {}
    for p in grupo:
        pub_p = p.get("publication") or {}
        for campo in ("category", "isbn", "issn", "publisher", "title"):
            if not pub_base.get(campo) and pub_p.get(campo):
                pub_base[campo] = pub_p[campo]
    base["publication"] = pub_base

    return base

# Remove artigos duplicados
def deduplicar(papers: list[dict]) -> list[dict]:
    # --- passo 1: agrupar por DOI ---
    grupos_doi = {}
    sem_doi = []
    for p in papers:
        doi = (p.get("doi") or "").strip().lower()
        if doi:
            grupos_doi.setdefault(doi, []).append(p)
        else:
            sem_doi.append(p)

    #print(grupos_doi)
    mesclados = [mesclar_grupo(g) for g in grupos_doi.values()]

    # --- passo 2: entre os sem DOI (+ os já mesclados, para pegar duplicata
    #     de um artigo com DOI em uma base e sem DOI em outra), fuzzy por título ---
    candidatos = mesclados + sem_doi
    titulos = [(c.get("title") or "").strip().lower() for c in candidatos]

    descartados = set()
    resultado = []
    for i in range(len(candidatos)):
        if i in descartados:
            continue
        grupo_atual = [candidatos[i]]
        for j in range(i + 1, len(candidatos)):
            if j in descartados:
                continue
            if not titulos[i] or not titulos[j]:
                continue
            score = fuzz.token_sort_ratio(titulos[i], titulos[j])
            if score >= LIMIAR_SIMILARIDADE:
                grupo_atual.append(candidatos[j])
                descartados.add(j)
        resultado.append(mesclar_grupo(grupo_atual) if len(grupo_atual) > 1 else grupo_atual[0])

    return resultado

# Montar matriz com a interseção de artigos por base
def montar_matriz_intersecao(papers: list[dict], bases: list[str]) -> pd.DataFrame:
    matriz = pd.DataFrame(0, index=bases, columns=bases)
    for p in papers:
        dbs_do_artigo = [db for db in (p.get("databases") or []) if db in bases]
        for b in dbs_do_artigo:
            matriz.loc[b, b] += 1  # diagonal = total de artigos únicos que incluem essa base
        for b1, b2 in combinations(sorted(set(dbs_do_artigo)), 2):
            matriz.loc[b1, b2] += 1
            matriz.loc[b2, b1] += 1
    return matriz

# Salva Json atualizado dos artigos
def save_json_articles(dados_original: dict, papers_dedup: list[dict], caminho_saida: str):
    dados = dict(dados_original)
    dados["papers"] = papers_dedup
    dados["number_of_papers"] = len(papers_dedup)
    # number_of_papers_by_database é mantido como veio (contagem original por base, pré-dedup)
    with open(caminho_saida, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=2)
    print(f"JSON atualizado salvo em: {caminho_saida}")

# Salvar Excel atualizado dos artigos
def save_excel_articles(dados: dict, matriz: pd.DataFrame, caminho_saida: str, caminho_png: str):
    papers = dados["papers"]
    linhas = []
    for p in papers:
        linhas.append({
            "titulo": p["title"],
            "abstract": p["abstract"],
            "autores": "; ".join(p["authors"]) if p["authors"] else "",
            "keywords": "; ".join(p["keywords"]) if p["keywords"] else "",
            "doi": p["doi"],
            "revista_evento": (p.get("publication") or {}).get("title"),
            "data_publicacao": p["publication_date"],
            "bases_onde_foi_encontrado": ", ".join(p["databases"]),
            "encontrado_em_mais_de_1_base": len(p["databases"]) > 1,
            "url": p["urls"][0] if p["urls"] else "",
        })
    df = pd.DataFrame(linhas)

    wb = Workbook()

    ws_dados = wb.active
    ws_dados.title = "Artigos"
    ws_dados.append(list(df.columns))
    for cell in ws_dados[1]:
        cell.font = Font(bold=True)
    destaque = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
    for row in df.itertuples(index=False):
        ws_dados.append(list(row))
        if row.encontrado_em_mais_de_1_base:
            for cell in ws_dados[ws_dados.max_row]:
                cell.fill = destaque
    larguras = [50, 60, 30, 30, 22, 30, 16, 25, 22, 40]
    for i, w in enumerate(larguras, start=1):
        ws_dados.column_dimensions[ws_dados.cell(row=1, column=i).column_letter].width = w

    ws_resumo = wb.create_sheet("Resumo")
    ws_resumo.append(["Base", "Nº de artigos (busca original)"])
    for cell in ws_resumo[1]:
        cell.font = Font(bold=True)
    contagem_original = dados["number_of_papers_by_database"]
    for base, qtd in contagem_original.items():
        ws_resumo.append([base, qtd])
    linha_total_orig = ws_resumo.max_row + 1
    ws_resumo.append(["TOTAL (com duplicatas entre bases)", sum(contagem_original.values())])
    ws_resumo.append(["TOTAL ÚNICO (após deduplicação)", dados["number_of_papers"]])
    ws_resumo["A1"].font = Font(bold=True)

    n = len(contagem_original)
    grafico = BarChart()
    grafico.title = "Artigos por base (busca original, antes do dedup)"
    grafico.y_axis.title = "Nº de artigos"
    dados_g = Reference(ws_resumo, min_col=2, min_row=1, max_row=1 + n)
    categorias_g = Reference(ws_resumo, min_col=1, min_row=2, max_row=1 + n)
    grafico.add_data(dados_g, titles_from_data=True)
    grafico.set_categories(categorias_g)
    ws_resumo.add_chart(grafico, "D2")

    ws_matriz = wb.create_sheet("Interseções")
    ws_matriz.append(["Base"] + list(matriz.columns))
    for cell in ws_matriz[1]:
        cell.font = Font(bold=True)
    for base_linha in matriz.index:
        ws_matriz.append([base_linha] + list(matriz.loc[base_linha]))
    n_bases = len(matriz)
    faixa_dados = f"B2:{ws_matriz.cell(row=1 + n_bases, column=1 + n_bases).coordinate}"
    ws_matriz.conditional_formatting.add(
        faixa_dados,
        ColorScaleRule(
            start_type="min", start_color="FFFFFF",
            end_type="max", end_color="4472C4",
        ),
    )
    ws_matriz["A1"].font = Font(bold=True)
    ws_matriz.column_dimensions["A"].width = 18
    for i in range(2, 2 + n_bases):
        ws_matriz.column_dimensions[ws_matriz.cell(row=1, column=i).column_letter].width = 18

    wb.save(caminho_saida)
    print(f"Excel atualizado salvo em: {caminho_saida}")

    # Heatmap standalone (PNG)
    plt.figure(figsize=(7.5, 6))
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar_kws={"label": "Nº de artigos em comum"})
    plt.title("Matriz de interseção entre bases (após deduplicação)")
    plt.tight_layout()
    plt.savefig(caminho_png, dpi=150)
    plt.close()
    print(f"Heatmap salvo em: {caminho_png}")

In [24]:
# Aplicação da remoção de artigos duplicados
dados = load_json_articles(os.path.join(PASTA_SAIDA, "unificado_findpapers.json"))
bases = list(dados["number_of_papers_by_database"].keys())

total_antes = len(dados["papers"])
papers_dedup = deduplicar(dados["papers"])
total_depois = len(papers_dedup)

print(f"Artigos antes da deduplicação: {total_antes}")
print(f"Artigos após deduplicação: {total_depois}")
print(f"Duplicatas removidas: {total_antes - total_depois}")

matriz = montar_matriz_intersecao(papers_dedup, bases)
print("\nMatriz de interseção:")
print(matriz)

save_json_articles(dados, papers_dedup, os.path.join(PASTA_SAIDA, "unificado_findpapers_dedup.json"))
save_excel_articles(
    {**dados, "papers": papers_dedup, "number_of_papers": total_depois},
    matriz,
    os.path.join(PASTA_SAIDA, "unificado_dedup.xlsx"),
    os.path.join(PASTA_SAIDA, "matriz_intersecao.png"),
)

Artigos antes da deduplicação: 909
Artigos após deduplicação: 545
Duplicatas removidas: 364

Matriz de interseção:
                Scopus  Web of Science  Embase  IEEE Xplore  PubMed
Scopus             469             164     153           23      21
Web of Science     164             226     129           19      10
Embase             153             129     160            7      11
IEEE Xplore         23              19       7           23       0
PubMed              21              10      11            0      30
JSON atualizado salvo em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados\unificado_findpapers_dedup.json
Excel atualizado salvo em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados\unificado_dedup.xlsx
Heatmap salvo em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados\matriz_intersecao.png


### **Filtro por Ano**

In [25]:
def extrair_ano(publication_date: str | None) -> int | None:
    if not publication_date:
        return None
    try:
        return int(publication_date[:4])
    except (ValueError, TypeError):
        return None
 
def filtrar_por_ano(papers: list[dict], ano_corte: int, manter_sem_ano: bool) -> tuple[list[dict], list[dict]]:
    mantidos, removidos = [], []
    for p in papers:
        ano = extrair_ano(p.get("publication_date"))
        if ano is None:
            (mantidos if manter_sem_ano else removidos).append(p)
        elif ano >= ano_corte:
            mantidos.append(p)
        else:
            removidos.append(p)
    return mantidos, removidos

def atualizar_json(caminho_json: str, mantidos: list[dict]):
    with open(caminho_json, "r", encoding="utf-8") as f:
        dados = json.load(f)
    dados["papers"] = mantidos
    dados["number_of_papers"] = len(mantidos)
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=2)
    print(f"JSON atualizado salvo em: {caminho_json}")

def atualizar_excel(caminho_excel: str, mantidos: list[dict], total_removidos: int, ano_corte: int):
    wb = load_workbook(caminho_excel)
    ws = wb["Artigos"]
 
    cabecalho = [c.value for c in ws[1]]
    if "ano_identificado" not in cabecalho:
        cabecalho.append("ano_identificado")
        ws.cell(row=1, column=len(cabecalho), value="ano_identificado").font = Font(bold=True)
    idx_ano_identificado = cabecalho.index("ano_identificado") + 1
    idx_data_pub = cabecalho.index("data_publicacao") + 1 if "data_publicacao" in cabecalho else None
 
    # limpa as linhas de dados atuais (mantendo o cabeçalho)
    ws.delete_rows(2, ws.max_row)
 
    destaque_sem_ano = PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid")
    for p in mantidos:
        ano = extrair_ano(p.get("publication_date"))
        linha = [
            p["title"],
            p["abstract"],
            "; ".join(p["authors"]) if p["authors"] else "",
            "; ".join(p["keywords"]) if p["keywords"] else "",
            p["doi"],
            (p.get("publication") or {}).get("title"),
            p["publication_date"],
            ", ".join(p["databases"]),
            len(p["databases"]) > 1,
            ano is not None,
        ]
        ws.append(linha)
        if ano is None:
            for cell in ws[ws.max_row]:
                cell.fill = destaque_sem_ano
 
    if "Resumo" in wb.sheetnames:
        ws_resumo = wb["Resumo"]
        linha_nova = ws_resumo.max_row + 2
        ws_resumo.cell(row=linha_nova, column=1, value=f"Filtro aplicado: ano >= {ano_corte}").font = Font(bold=True)
        ws_resumo.cell(row=linha_nova + 1, column=1, value="Artigos removidos pelo filtro de ano")
        ws_resumo.cell(row=linha_nova + 1, column=2, value=total_removidos)
        ws_resumo.cell(row=linha_nova + 2, column=1, value="TOTAL final (após dedup + filtro de ano)")
        ws_resumo.cell(row=linha_nova + 2, column=2, value=len(mantidos))
 
    wb.save(caminho_excel)
    print(f"Excel atualizado salvo em: {caminho_excel}")

In [26]:
caminho_json = "unificado_findpapers_dedup.json"
caminho_excel = "unificado_dedup.xlsx"
 
with open(os.path.join(PASTA_SAIDA,caminho_json), "r", encoding="utf-8") as f:
    dados = json.load(f)

total_antes = len(dados["papers"])
mantidos, removidos = filtrar_por_ano(dados["papers"], ANO_CORTE, MANTER_SEM_ANO)

sem_ano = [p for p in dados["papers"] if extrair_ano(p.get("publication_date")) is None]

print(f"Artigos antes do filtro: {total_antes}")
print(f"Artigos removidos (ano < {ANO_CORTE}): {len(removidos) if not MANTER_SEM_ANO else len([p for p in removidos if extrair_ano(p.get('publication_date')) is not None])}")
print(f"Artigos sem ano identificado: {len(sem_ano)} ({'mantidos' if MANTER_SEM_ANO else 'removidos'})")
print(f"Artigos após o filtro: {len(mantidos)}")

if removidos:
    print("\nExemplos de artigos removidos:")
    for p in removidos[:5]:
        print(f" - [{extrair_ano(p.get('publication_date'))}] {p['title'][:70]}")

atualizar_json(os.path.join(PASTA_SAIDA,caminho_json), mantidos)
atualizar_excel(os.path.join(PASTA_SAIDA,caminho_excel), mantidos, len(removidos), ANO_CORTE)

Artigos antes do filtro: 545
Artigos removidos (ano < 2015): 23
Artigos sem ano identificado: 0 (mantidos)
Artigos após o filtro: 522

Exemplos de artigos removidos:
 - [2007] Fluoroscopic tracking of multiple implanted fiducial markers using mul
 - [2013] Automatic inference and measurement of 3D carpal bone kinematics from 
 - [2006] Semiautomatic 3-D prostate segmentation from TRUS images using spheric
 - [2009] Fluoroscopic tumor tracking for image-guided lung cancer radiotherapy
 - [2008] A robust and accurate two-stage approach for automatic recovery of dis
JSON atualizado salvo em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados\unificado_findpapers_dedup.json
Excel atualizado salvo em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados\unificado_dedup.xlsx


### **Visualização Exploratória Metadados de Artigos**

In [27]:
# Visualização Exploratória dos artigos
def carregar_papers_analise_exploratoria(caminho_json: str) -> pd.DataFrame:
    with open(caminho_json, "r", encoding="utf-8") as f:
        dados = json.load(f)
 
    linhas = []
    for p in dados["papers"]:
        ano = None
        if p.get("publication_date"):
            try:
                ano = int(p["publication_date"][:4])
            except (ValueError, TypeError):
                ano = None
 
        dbs = p.get("databases") or ["(desconhecida)"]
        if BASE_PRIMARIA:
            linhas.append({"titulo": p["title"], "ano": ano, "base": dbs[0]})
        else:
            for db in dbs:
                linhas.append({"titulo": p["title"], "ano": ano, "base": db})
 
    df = pd.DataFrame(linhas)
    n_sem_ano = df["ano"].isna().sum()
    if n_sem_ano:
        print(f"[aviso] {n_sem_ano} artigo(s) sem ano identificável (não entram no gráfico)")
    return df.dropna(subset=["ano"])

def montar_tabela_pivot(df: pd.DataFrame) -> pd.DataFrame:
    tabela = df.pivot_table(index="ano", columns="base", values="titulo", aggfunc="count", fill_value=0)
    tabela = tabela.sort_index()
    tabela.index = tabela.index.astype(int)
    return tabela

def plotar_grafico(tabela: pd.DataFrame, caminho_png: str):
    cores = sns.color_palette("tab10", n_colors=tabela.shape[1])
    ax = tabela.plot(kind="bar", stacked=True, figsize=(10, 6), color=cores)
    titulo_extra = "1 artigo = 1 base (primária)" if BASE_PRIMARIA else "artigo pode aparecer em +1 base"
    ax.set_title(f"Frequência de publicações por ano e por base ({titulo_extra})")
    ax.set_xlabel("Ano de publicação")
    ax.set_ylabel("Nº de artigos")
    ax.legend(title="Base", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(caminho_png, dpi=150)
    plt.close()
    print(f"Gráfico salvo em: {caminho_png}")

def adicionar_ao_excel(tabela: pd.DataFrame, caminho_excel: str):
    wb = load_workbook(caminho_excel)
    nome_aba = "Frequência Ano-Base"
    if nome_aba in wb.sheetnames:
        del wb[nome_aba]
    ws = wb.create_sheet(nome_aba)
 
    ws.append(["Ano"] + list(tabela.columns))
    for cell in ws[1]:
        cell.font = Font(bold=True)
    for ano, row in tabela.iterrows():
        ws.append([ano] + list(row))
 
    n_linhas = tabela.shape[0]
    n_bases = tabela.shape[1]
 
    grafico = BarChart()
    grafico.type = "col"
    grafico.grouping = "stacked"
    grafico.overlap = 100
    grafico.title = "Frequência de publicações por ano e por base"
    grafico.y_axis.title = "Nº de artigos"
    grafico.x_axis.title = "Ano"
    dados_g = Reference(ws, min_col=2, max_col=1 + n_bases, min_row=1, max_row=1 + n_linhas)
    categorias_g = Reference(ws, min_col=1, min_row=2, max_row=1 + n_linhas)
    grafico.add_data(dados_g, titles_from_data=True)
    grafico.set_categories(categorias_g)
    grafico.width = 24
    grafico.height = 12
    ws.add_chart(grafico, f"{chr(65 + n_bases + 2)}2")
 
    wb.save(caminho_excel)
    print(f"Aba '{nome_aba}' adicionada em: {caminho_excel}")

def carregar_papers_completo(caminho_json: str) -> tuple[pd.DataFrame, list[dict]]:
    """Retorna o DataFrame usado no gráfico ano+base e também a lista crua de papers,
    para os gráficos de revista/DOI/abstract (que não dependem de ano)."""
    with open(caminho_json, "r", encoding="utf-8") as f:
        dados = json.load(f)
    return dados["papers"]

def montar_tabela_metadados(papers: list[dict]) -> pd.DataFrame:
    linhas = []
    for p in papers:
        revista = (p.get("publication") or {}).get("title") or "(revista/evento não identificado)"
        linhas.append({
            "titulo": p["title"],
            "revista": revista,
            "tem_doi": bool(p.get("doi")),
            "tem_abstract": bool(p.get("abstract")),
        })
    return pd.DataFrame(linhas)

def plot_frequency_database(df: pd.DataFrame, path: str):
    contagem = df["base"].value_counts().sort_values(ascending=False)

    plt.figure(figsize=(8, 5))
    ax = contagem.plot(kind="bar", color="#4472C4")
    ax.set_title("Frequência de artigos por base")
    ax.set_xlabel("Base")
    ax.set_ylabel("Nº de artigos")
    for i, v in enumerate(contagem.values):
        ax.text(i, v, str(v), ha="center", va="bottom")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close()
    return contagem

def plotar_top_revistas(df_meta: pd.DataFrame, caminho_png: str, top_n: int = 15, largura_max_linha: int = 80):
    contagem = df_meta["revista"].value_counts().head(top_n).sort_values()
    rotulos_quebrados = [textwrap.fill(nome, largura_max_linha) for nome in contagem.index]

    fig, ax = plt.subplots(figsize=(11, max(4, 0.55 * len(contagem))))
    ax.barh(rotulos_quebrados, contagem.values, color="#4472C4")
    ax.set_title(f"Top {top_n} revistas/eventos com mais artigos")
    ax.set_xlabel("Nº de artigos")
    for i, v in enumerate(contagem.values):
        ax.text(v, i, f" {v}", va="center")
    ax.margins(x=0.08)
    plt.tight_layout()
    plt.savefig(caminho_png, dpi=150)
    plt.close()
    print(f"Gráfico salvo em: {caminho_png}")
    
def plotar_percentual_completude(df_meta: pd.DataFrame, campo: str, rotulo: str, caminho_png: str):
    total = len(df_meta)
    com = int(df_meta[campo].sum())
    sem = total - com
    pct_com = 100 * com / total if total else 0
    pct_sem = 100 * sem / total if total else 0
 
    plt.figure(figsize=(5, 5))
    barras = plt.bar([f"Com {rotulo}", f"Sem {rotulo}"], [com, sem],
                      color=["#4472C4", "#C00000"])
    plt.title(f"Completude de {rotulo} ({total} artigos)")
    plt.ylabel("Nº de artigos")
    for barra, pct in zip(barras, [pct_com, pct_sem]):
        altura = barra.get_height()
        plt.text(barra.get_x() + barra.get_width() / 2, altura,
                  f"{altura:.0f}\n({pct:.1f}%)", ha="center", va="bottom")
    plt.tight_layout()
    plt.ylim(0, 1.1 * max(pct_com, pct_sem) * total / 100)
    plt.savefig(caminho_png, dpi=150)
    plt.close()
    print(f"Gráfico salvo em: {caminho_png} | Com {rotulo}: {pct_com:.1f}% | Sem {rotulo}: {pct_sem:.1f}%")

def adicionar_metadados_ao_excel(df_meta: pd.DataFrame, caminho_excel: str, top_n: int = 15):
    wb = load_workbook(caminho_excel)
    nome_aba = "Revistas e Completude"
    if nome_aba in wb.sheetnames:
        del wb[nome_aba]
    ws = wb.create_sheet(nome_aba)
 
    ws.append(["Top revistas/eventos"])
    ws["A1"].font = Font(bold=True)
    ws.append(["Revista/Evento", "Nº de artigos"])
    for cell in ws[2]:
        cell.font = Font(bold=True)
    contagem = df_meta["revista"].value_counts().head(top_n)
    linha_inicio_revistas = 3
    for revista, qtd in contagem.items():
        ws.append([revista, qtd])
    linha_fim_revistas = linha_inicio_revistas + len(contagem) - 1
 
    grafico_revistas = BarChart()
    grafico_revistas.title = f"Top {top_n} revistas/eventos"
    grafico_revistas.type = "bar"  # barras horizontais
    dados_g = Reference(ws, min_col=2, min_row=2, max_row=linha_fim_revistas)
    categorias_g = Reference(ws, min_col=1, min_row=linha_inicio_revistas, max_row=linha_fim_revistas)
    grafico_revistas.add_data(dados_g, titles_from_data=True)
    grafico_revistas.set_categories(categorias_g)
    grafico_revistas.height = 12
    grafico_revistas.width = 18
    ws.add_chart(grafico_revistas, "D2")
 
    linha_atual = linha_fim_revistas + 3
    total = len(df_meta)
    for campo, rotulo in (("tem_doi", "DOI"), ("tem_abstract", "Abstract")):
        com = int(df_meta[campo].sum())
        sem = total - com
        ws.cell(row=linha_atual, column=1, value=f"Completude de {rotulo}").font = Font(bold=True)
        linha_atual += 1
        ws.cell(row=linha_atual, column=1, value="Categoria").font = Font(bold=True)
        ws.cell(row=linha_atual, column=2, value="Nº de artigos").font = Font(bold=True)
        ws.cell(row=linha_atual, column=3, value="%").font = Font(bold=True)
        linha_cab = linha_atual
        linha_atual += 1
        ws.cell(row=linha_atual, column=1, value=f"Com {rotulo}")
        ws.cell(row=linha_atual, column=2, value=com)
        ws.cell(row=linha_atual, column=3, value=round(100 * com / total, 1) if total else 0)
        linha_atual += 1
        ws.cell(row=linha_atual, column=1, value=f"Sem {rotulo}")
        ws.cell(row=linha_atual, column=2, value=sem)
        ws.cell(row=linha_atual, column=3, value=round(100 * sem / total, 1) if total else 0)
        linha_fim = linha_atual
 
        grafico_pct = BarChart()
        grafico_pct.title = f"Completude de {rotulo}"
        dados_g = Reference(ws, min_col=2, min_row=linha_cab, max_row=linha_fim)
        categorias_g = Reference(ws, min_col=1, min_row=linha_cab + 1, max_row=linha_fim)
        grafico_pct.add_data(dados_g, titles_from_data=True)
        grafico_pct.set_categories(categorias_g)
        grafico_pct.height = 7
        grafico_pct.width = 10
        ws.add_chart(grafico_pct, f"E{linha_cab}")
 
        linha_atual = linha_fim + 3
 
    ws.column_dimensions["A"].width = 55
    ws.column_dimensions["B"].width = 14
    ws.column_dimensions["C"].width = 10
 
    wb.save(caminho_excel)
    print(f"Aba '{nome_aba}' adicionada em: {caminho_excel}")


In [28]:
# Montagem e Salvamento dos Gráficos.
df = carregar_papers_analise_exploratoria(os.path.join(PASTA_SAIDA, "unificado_findpapers_dedup.json"))
plot_frequency_database(df, os.path.join(PASTA_SAIDA, "contagem_por_base_dedup.png"))
tabela = montar_tabela_pivot(df)
print("\nTabela ano x base:")
print(tabela)

plotar_grafico(tabela, os.path.join(PASTA_SAIDA, "frequencia_ano_base.png"))
adicionar_ao_excel(tabela, os.path.join(PASTA_SAIDA, "unificado.xlsx"))

papers = carregar_papers_completo(os.path.join(PASTA_SAIDA, "unificado_findpapers_dedup.json"))
df_meta = montar_tabela_metadados(papers)

plotar_top_revistas(df_meta, os.path.join(PASTA_SAIDA, "frequencia_revistas.png"))
plotar_percentual_completude(df_meta, "tem_doi", "DOI", os.path.join(PASTA_SAIDA, "completude_doi.png"))
plotar_percentual_completude(df_meta, "tem_abstract", "Abstract", os.path.join(PASTA_SAIDA, "completude_abstract.png"))
adicionar_metadados_ao_excel(df_meta, os.path.join(PASTA_SAIDA, "unificado_dedup.xlsx"))


Tabela ano x base:
base  Embase  PubMed  Scopus  Web of Science
ano                                         
2015       0       0       3               0
2016       0       0       2               0
2017       0       0       3               0
2018       0       0      10               6
2019       0       0      13               8
2020       0       0      22               5
2021       0       0      41               9
2022       0       0      40               5
2023       1       0      48               4
2024       2       2      85              12
2025       1       2     109               5
2026       1       3      75               5
Gráfico salvo em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados\frequencia_ano_base.png
Aba 'Frequência Ano-Base' adicionada em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\INCA\vfss_to_docker\data\artigos\metadados\unificado.xlsx
Gráfico salvo em: C:\Users\vinic\OneDrive\Documentos\0300_Projetos\I